# 03-01 常用损失函数

损失函数的作用是衡量模型预测错得有多严重。

但不同任务里，“错”的含义不一样：

- 回归任务关心预测数值和真实数值差多少。
- 分类任务关心模型有没有把概率给到正确类别。
- 类别不平衡任务还要特别关心少数类有没有被忽略。
- 分割任务关心预测区域和真实区域重合多少。

所以损失函数不是随便选的，它必须和任务目标匹配。

## 1. 先按任务类型建立地图

| 任务类型 | 常用损失函数 | 典型用途 |
|---|---|---|
| 回归 | MSE、MAE、Huber、Smooth L1 | 房价预测、温度预测、连续值预测 |
| 二分类 | Binary Cross Entropy | 是否患病、是否垃圾邮件 |
| 多分类 | Cross Entropy | MNIST、图像类别识别 |
| 多标签分类 | 多个 Binary Cross Entropy | 一张图同时有猫、沙发、窗户 |
| 类别不平衡分类 | Focal Loss、加权交叉熵 | 正负样本极不均衡 |
| 概率分布对齐 | KL Divergence | 蒸馏、生成模型、分布匹配 |
| 图像分割 | Dice Loss、IoU Loss | 医学图像分割、目标区域分割 |

初学阶段先重点掌握：MSE、MAE、Huber、Smooth L1、BCE、Cross Entropy。

## 2. MSE 均方误差

MSE 的英文是 Mean Squared Error，常用于回归任务。

单个样本的平方误差是：

$$
(\hat{y}-y)^2
$$

多个样本取平均：

$$
\mathcal{L}_{MSE}=\frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})^2
$$

它在惩罚什么？

MSE 惩罚预测值和真实值之间的距离，而且大错误会被平方放大。

例如误差是 $2$，平方后是 $4$；误差是 $10$，平方后是 $100$。

所以 MSE 对大错误非常敏感。

## 3. MSE 的优缺点

MSE 的优点：

- 公式简单。
- 可导，优化方便。
- 会强烈惩罚大错误。

MSE 的缺点：

- 对异常值很敏感。
- 如果数据里有极端错误标签，模型可能被这些异常点带偏。

适用场景：

当你希望模型特别重视大误差时，可以用 MSE。

例如房价预测中，预测偏差很大通常比小偏差严重得多，MSE 就比较合理。

## 4. MAE 平均绝对误差

MAE 的英文是 Mean Absolute Error。

单个样本的绝对误差是：

$$
|\hat{y}-y|
$$

多个样本取平均：

$$
\mathcal{L}_{MAE}=\frac{1}{m}\sum_{i=1}^{m}|\hat{y}^{(i)}-y^{(i)}|
$$

它在惩罚什么？

MAE 只看预测值和真实值相差多少，不会像 MSE 那样把大误差平方放大。

误差是 $2$，损失就是 $2$；误差是 $10$，损失就是 $10$。

## 5. MAE 的优缺点

MAE 的优点：

- 对异常值更稳健。MSE的平方放大机制会让异常值对模型影响过大，而 MAE 不会。
- 损失值和原始误差单位一致，比较直观。

MAE 的缺点：

- 在 $\hat{y}=y$ 处不可导。
- 梯度大小基本恒定，离目标很远时也不会像 MSE 那样给更强的修正信号。

适用场景：

当数据里可能有异常值，且不希望模型被少数极端样本强烈影响时，可以考虑 MAE。

## 6. Huber Loss

Huber Loss 可以理解成 MSE 和 MAE 的折中。

当误差比较小时，它像 MSE；当误差比较大时，它像 MAE。

设误差为：

$$
e=\hat{y}-y
$$

Huber Loss 定义为：

$$
\mathcal{L}_{\delta}(e)=
\begin{cases}
\frac{1}{2}e^2, & |e|\le \delta \\
\delta(|e|-\frac{1}{2}\delta), & |e|>\delta
\end{cases}
$$

$\delta$ 是分界点。

它在惩罚什么？

- 小误差：用平方惩罚，让模型精细修正。
- 大误差：用近似线性惩罚，避免异常值影响太大。

## 7. Huber Loss 什么时候用

Huber Loss 的优点：

- 比 MSE 更抗异常值。
- 比 MAE 更平滑，优化更舒服。

缺点：

- 多了一个超参数 $\delta$。
- $\delta$ 选得不合适时，效果可能不理想。

适用场景：

回归任务中，如果你既想保留 MSE 的平滑优化，又担心异常值，就可以考虑 Huber Loss。

## 7.5 Smooth L1 Loss

Smooth L1 Loss 也常用于回归任务，尤其常见于目标检测里的边界框回归。

它和 Huber Loss 很像，也是在小误差时像 MSE，大误差时像 MAE。

先设误差为：

$$
e=\hat{y}-y
$$

常见的 Smooth L1 定义是：

$$
\mathcal{L}_{SmoothL1}(e)=
\begin{cases}
\frac{1}{2}e^2, & |e|<1 \\
|e|-\frac{1}{2}, & |e|\ge 1
\end{cases}
$$

这个公式的意思是：

- 当误差很小，使用平方项，曲线平滑，方便模型细致修正。
- 当误差很大，使用近似线性项，避免异常值像 MSE 那样被平方放大。

所以 Smooth L1 可以理解成：**比 MAE 更平滑，比 MSE 更抗异常值。**

### Smooth L1 和 Huber 的关系

Huber Loss 通常写成带分界参数 $\delta$ 的形式：

$$
\mathcal{L}_{\delta}(e)=
\begin{cases}
\frac{1}{2}e^2, & |e|\le \delta \\
\delta(|e|-\frac{1}{2}\delta), & |e|>\delta
\end{cases}
$$

Smooth L1 可以看成一种常用的 Huber 风格损失。很多框架会用参数 $\beta$ 控制小误差区域：

$$
\mathcal{L}_{SmoothL1,\beta}(e)=
\begin{cases}
\frac{1}{2}\frac{e^2}{\beta}, & |e|<\beta \\
|e|-\frac{1}{2}\beta, & |e|\ge \beta
\end{cases}
$$

$\beta$ 越大，使用平方惩罚的区域越宽；$\beta$ 越小，越接近 MAE 的线性惩罚。

### Smooth L1 什么时候用

Smooth L1 适合这些情况：

- 回归任务里可能有异常值。
- 希望小误差附近优化更平滑。
- 不希望大误差被 MSE 过度放大。
- 目标检测中做边界框位置回归。

它的直觉很实用：小错认真修，大错别被少数异常点带跑。

## 8. Binary Cross Entropy 二元交叉熵

Binary Cross Entropy，简称 BCE，用于二分类任务。

二分类中真实标签是：

$$
y\in\{0,1\}
$$

模型输出类别 $1$ 的概率：

$$
\hat{y}=P(y=1\mid x)
$$

BCE 公式是：

$$
\mathcal{L}_{BCE}=-\left[y\log(\hat{y})+(1-y)\log(1-\hat{y})\right]
$$

它在惩罚什么？

如果真实标签是 $1$，公式会变成：

$$
\mathcal{L}=-\log(\hat{y})
$$

这时模型给真实类别的概率越小，损失越大。

如果真实标签是 $0$，公式会变成：

$$
\mathcal{L}=-\log(1-\hat{y})
$$

这时模型越错误地相信类别 $1$，损失越大。

## 9. BCE 和 Sigmoid 的关系

二分类模型通常先输出一个 logit：

$$
z=\mathbf{w}^{T}\mathbf{x}+b
$$

再经过 Sigmoid 得到概率：

$$
\hat{y}=\sigma(z)
$$

然后用 BCE 比较 $\hat{y}$ 和 $y$。

所以二分类常见组合是：

```text
logit -> Sigmoid -> BCE
```

很多深度学习框架会提供数值更稳定的组合版本，直接接收 logits。概念上你先理解：BCE 惩罚的是模型没有把高概率给真实二分类标签。

## 10. Cross Entropy 多分类交叉熵

多分类任务中，每个样本属于 $C$ 个类别中的一个。

模型输出 logits：

$$
\mathbf{z}=[z_1,z_2,\dots,z_C]
$$

Softmax 把 logits 变成概率：

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^{C}e^{z_j}}
$$

如果真实类别是第 $t$ 类，交叉熵损失是：

$$
\mathcal{L}_{CE}=-\log(p_t)
$$

这句话很直观：真实类别的预测概率越大，损失越小；真实类别的预测概率越小，损失越大。

大白话解释：

$$
损失函数的结果 = 预测正确的类别的概率的对数的负数
$$

## 11. 多分类交叉熵为什么常用

多分类交叉熵的优点：

- 直接对应“把概率给正确类别”这个目标。
- 和 Softmax 配合后梯度形式简洁。
- 对错误且自信的预测惩罚很大。

Softmax + Cross Entropy 的梯度有一个重要结论：

$$
\frac{\partial \mathcal{L}}{\partial z_i}=p_i-y_i
$$

其中 $y_i$ 是 one-hot 标签。

如果模型给错误类别的概率太高，$p_i-y_i$ 就会推动这个类别的 logit 降低。

如果真实类别概率太低，$p_i-y_i$ 会推动真实类别的 logit 升高。

## 12. NLL Loss 负对数似然损失

NLL 的英文是 Negative Log Likelihood。

如果模型已经输出了每个类别的对数概率：

$$
\log p_i
$$

真实类别是第 $t$ 类，那么 NLL Loss 是：

$$
\mathcal{L}_{NLL}=-\log p_t
$$

所以 NLL 和交叉熵关系很近。

可以粗略记成：

```text
Cross Entropy = LogSoftmax + NLLLoss
```

它们本质上都在惩罚真实类别的概率不够高。

## 13. Hinge Loss

Hinge Loss 常见于支持向量机，也可以用于分类任务。

二分类中常把标签写成：

$$
y\in\{-1,1\}
$$

模型输出分数 $f(x)$。

Hinge Loss 是：

$$
\mathcal{L}_{hinge}=\max(0,1-yf(x))
$$

它在惩罚什么？

它不仅希望模型分类正确，还希望正确得有一定距离。

如果：

$$
yf(x)\ge 1
$$

损失就是 $0$。

这表示模型不仅分对了，而且留出了足够间隔。

## 14. KL Divergence KL 散度

KL 散度用于衡量两个概率分布的差异。

假设真实分布是 $P$，模型分布是 $Q$，KL 散度是：

$$
D_{KL}(P\|Q)=\sum_i P(i)\log\frac{P(i)}{Q(i)}
$$

它在惩罚什么？

如果模型分布 $Q$ 和目标分布 $P$ 差得很远，KL 散度就大。

如果 $Q$ 很接近 $P$，KL 散度就小。

常见场景：

- 知识蒸馏：学生模型模仿教师模型的概率分布。
- 生成模型：让某个分布接近目标分布。
- 分布匹配：不仅关心正确类别，还关心整体概率形状。

## 15. Focal Loss

Focal Loss 常用于类别不平衡问题。

比如目标检测中，大量样本是背景，真正的目标样本很少。

普通交叉熵可能被大量容易分类的样本主导，模型对少数难样本关注不够。

Focal Loss 的二分类形式可以写成：

$$
\mathcal{L}_{focal}=-\alpha(1-p_t)^\gamma\log(p_t)
$$

其中 $p_t$ 是模型给真实类别的概率。

它的核心是这一项：

$$
(1-p_t)^\gamma
$$

如果样本很容易，$p_t$ 很大，这一项会很小，损失被压低。

如果样本很难，$p_t$ 很小，这一项不会太小，模型会继续关注它。

## 16. Dice Loss

Dice Loss 常用于图像分割，尤其是医学图像分割。

分割任务关心的是预测区域和真实区域重合多少。

Dice 系数是：

$$
\operatorname{Dice}=\frac{2|A\cap B|}{|A|+|B|}
$$

其中：

- $A$ 是预测区域。
- $B$ 是真实区域。
- $A\cap B$ 是两者重叠区域。

Dice Loss 通常写成：

$$
\mathcal{L}_{Dice}=1-\operatorname{Dice}
$$

重合越多，Dice 越接近 $1$，损失越接近 $0$。

## 17. IoU Loss

IoU 的英文是 Intersection over Union，交并比。

它也是衡量两个区域重合程度的指标：

$$
\operatorname{IoU}=\frac{|A\cap B|}{|A\cup B|}
$$

IoU Loss 可以写成：

$$
\mathcal{L}_{IoU}=1-\operatorname{IoU}
$$

它和 Dice Loss 都常用于分割任务。

直觉区别是：Dice 更强调重叠部分，IoU 用交集除以并集，惩罚预测区域和真实区域整体不一致。

## 18. 损失函数怎么选

初学阶段可以按任务类型来选：

| 任务 | 首选损失 |
|---|---|
| 普通回归 | MSE |
| 回归且异常值较多 | MAE、Huber 或 Smooth L1 |
| 目标检测边界框回归 | Smooth L1 或 IoU 系列损失 |
| 二分类 | BCE |
| 多分类 | Cross Entropy |
| 多标签分类 | 每个标签一个 BCE |
| 类别严重不平衡 | 加权交叉熵或 Focal Loss |
| 分布匹配 | KL Divergence |
| 图像分割 | Dice Loss、IoU Loss、交叉熵组合 |

不要先背所有公式。先问自己：这个任务里，模型到底应该被惩罚什么。

## 19. 常见选择错误

第一，二分类和多分类混淆。

二分类通常是一个输出表示类别 $1$ 的概率。

多分类通常是 $C$ 个 logits，表示 $C$ 个类别分数。

第二，多分类和多标签混淆。

多分类是多个类别选一个，适合 Softmax + Cross Entropy。

多标签是多个标签可以同时成立，适合多个 Sigmoid + BCE。

第三，回归任务乱用分类损失。

如果目标是连续数值，就应该先考虑 MSE、MAE、Huber，而不是交叉熵。

## 20. 本节总结

这一节的逻辑链是：

```text
损失函数衡量模型错在哪里
-> 不同任务对“错”的定义不同
-> 回归关心数值距离
-> 分类关心真实类别概率
-> 类别不平衡关心难样本和少数类
-> 分割关心预测区域和真实区域重合程度
```

先重点掌握五个：

- MSE：普通回归。
- MAE：对异常值更稳健的回归。
- Huber：MSE 和 MAE 的折中。
- Smooth L1：Huber 风格损失，小误差平滑，大误差稳健。
- BCE：二分类。
- Cross Entropy：多分类。

真正选择损失函数时，不要只问“别人常用什么”，要问：我的任务希望模型为什么行为付出代价。